# Network Creation and analysis

The following Notebook contains the code used for the creation of the lexical dynamic network and its subsequent analyses. The **dynetx** library is requested.

In [1]:
import networkx as nx
import pandas as pd
import pyphen
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
import os

## 1. Dynamic graph creation

The code extracts the nodes and the snapshot they belong to from the text embedding created in the ```embedding``` folder. Specifically, the requested structure to create the dynamic network is the following:
```
n1 n2 t1
```
where
- ```n1``` and ```n2``` are nodes
- ```t1``` is the timestamp of interaction appearance


In [ ]:
# CONFIGURAZIONE
input_dir = Path("../edges_analysis/edges")  
graphs = {} # Dizionario per salvare i grafi: chiave=nome_file, valore=oggetto Graph

# Verifica che la cartella esista
if not input_dir.exists():
    print(f"Errore: La cartella {input_dir} non esiste.")
else:
    # Itera su tutti i file .tsv nella cartella
    for file_path in input_dir.glob("*.tsv"):
        
        print(f"Elaborazione: {file_path.name}...", end="")
        
        try:
            # 1. Leggi il TSV con Pandas
            # Assumo che le colonne si chiamino 'source', 'target', 'weight' o simili.
            # Se il file NON ha header, aggiungi header=None e names=['u', 'v', 'w']
            df = pd.read_csv(file_path, sep='\t')
            
            # --- CONTROLLO NOMI COLONNE ---
            # Adatta questi nomi a quelli reali nel tuo file TSV
            # Esempio: se nel file si chiamano "Parola1", "Parola2", "Simil", cambiali qui sotto.
            col_source = df.columns[0]  # Prende la prima colonna come nodo i
            col_target = df.columns[1]  # Prende la seconda colonna come nodo j
            col_weight = df.columns[2]  # Prende la terza colonna come peso
            
            # 2. Crea il Grafo Pesato
            # create_using=nx.Graph() crea un grafo non direzionato (simmetrico)
            G = nx.from_pandas_edgelist(
                df, 
                source=col_source, 
                target=col_target, 
                edge_attr=col_weight,
                create_using=nx.Graph() 
            )
            
            # 3. Salva nel dizionario
            snapshot_name = file_path.stem # Es: "snap1960" da "snap1960.tsv"
            graphs[snapshot_name] = G
            
            print(f" Fatto. Nodi: {G.number_of_nodes()}, Archi: {G.number_of_edges()}")
            
            nx.write_gexf(G, f"./networks/{snapshot_name}.gexf")
        except Exception as e:
            print(f" Errore: {e}")

# --- ESEMPIO DI UTILIZZO ---
# Accesso al grafo del primo snapshot:
if graphs:
    first_key = list(graphs.keys())[0]
    G_test = graphs[first_key]
    
    # Verifica il peso di un arco a caso
    u, v = list(G_test.edges())[0]
    peso = G_test[u][v][col_weight] # O semplicemente G_test[u][v]
    print(f"\nEsempio arco in {first_key}: {u} <-> {v} con similarità {peso}")


# 2. Node Labelling
For the Complexity labelling, 3 metrics are merged:

1. **Morphological Complexity**: By the Zipf Law (or law of least effort) complex words are inveresely proportional to their length in terms of characters and syllables.
2. **De Mauro's list of common words**: common words can be Fundamentals (high frequency words, 90% coverage of texts), High Usage (6-8% coverage of texts) and High Availability (common but rarely used words). A complex word doesn't belong to any of these sets. 
3. **Inverse Document Frequency**: A generic/less complex word tends to appear more frequently than a complex/high informative word.

In [ ]:
# 1. Setup sillabatore italiano
dic = pyphen.Pyphen(lang='it_IT')

# 2. Caricamento lista De Mauro (Ground Truth)

nvdb = pd.read_csv('nvdb.csv').set_index('lemma')['categoria'].to_dict()

# 3. Funzione di calcolo complessità (parametri: parola, N documenti, freq doc parola)
def get_word_complexity(lemma, total_doc_count, doc_freq_of_lemma):
    """
    Restituisce un vettore di complessità per il lemma
    """
    
    # Metrica 1: Lunghezza
    char_len = len(lemma)
    syll_len = len(dic.inserted(lemma).split('-'))
    
    # Metrica 2: Categoria De Mauro (Score arbitrario per ordinamento)
    # 0=Fondamentale (semplice), 1=Alto Uso, 2=Alta Disp, 3=Altro (Complesso)
    cat = nvdb.get(lemma, 'Altro')
    cat_score = {'FO': 0, 'AU': 1, 'AD': 2, 'Altro': 3}.get(cat, 3)
    
    # Metrica 3: Informatività (IDF semplice)
    # Più alto è l'IDF, più la parola è specifica/rara (complessa)
    import math
    idf = math.log(total_doc_count / (1 + doc_freq_of_lemma))
    
    return {
        'char_len': char_len,
        'syll_len': syll_len,
        'demauro_level': cat_score,
        'idf': idf
    }

# Esempio di integrazione nel grafo
# G è il tuo grafo NetworkX/DyNetX
# for node in G.nodes():
#    attrs = get_word_complexity(node, N_DOCS, doc_freqs[node])
#    G.nodes[node].update(attrs)

In [ ]:
# qui salviamo il grafo dinamico in un file così da poterlo caricare in futuro senza dover rifare tutto

## 3. Network analysis

In [ ]:
# snapshot ids
times = sorted(g.temporal_snapshots_ids())
times

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

In [ ]:
# dynamic graph analysis
rows = []

for t in times:
    # snapshot al tempo t (intervallo [t, t])
    Gt = g.time_slice(t, t)

    num_nodes = Gt.number_of_nodes()
    num_edges = Gt.number_of_edges()

    density = nx.density(Gt) if num_nodes > 1 else 0

    avg_degree = (
        sum(dict(Gt.degree()).values()) / num_nodes
        if num_nodes > 0 else 0
    )

    if num_nodes > 0:
        num_components = nx.number_connected_components(Gt)
        largest_cc = max(len(c) for c in nx.connected_components(Gt))
    else:
        num_components = 0
        largest_cc = 0

    clustering = (
        nx.average_clustering(Gt)
        if num_edges > 0 else 0
    )

    rows.append({
        "time": t,
        "nodes": num_nodes,
        "edges": num_edges,
        "density": density,
        "avg_degree": avg_degree,
        "components": num_components,
        "largest_cc": largest_cc,
        "clustering": clustering
    })

pd.options.display.float_format = '{:.4f}'.format
pd.set_option('display.width', 1000)
df = pd.DataFrame(rows).sort_values("time")
print(df)


    time  nodes  edges  density  avg_degree  components  largest_cc  clustering
0      0     12     12   0.1818      2.0000           4           3      1.0000
1      1     15      9   0.0857      1.2000           7           3      0.2000
2      2     14     11   0.1209      1.5714           3           6      0.0000
3      3     18      9   0.0588      1.0000           9           2      0.0000
4      4     10     11   0.2444      2.2000           3           4      0.9333
5      5      8      8   0.2857      2.0000           1           8      0.0000
6      6     18      9   0.0588      1.0000           9           2      0.0000
7      7     18      9   0.0588      1.0000           9           2      0.0000
8      8      9      9   0.2500      2.0000           3           3      1.0000
9      9     12      6   0.0909      1.0000           6           2      0.0000
10    10      7      7   0.3333      2.0000           1           7      0.0000
11    11     12      6   0.0909      1.0

Creo il grafo con iGraph, mettendo come attributi i label della parola, le frequenze, e i pesi degli archi

In [ ]:
import pandas as pd
import igraph as ig
from pathlib import Path
import time

edges_dir = Path("../edges_analysis/edges")
vec_dir = Path("../embedding/word_embeddings_cleaned")
freq_dir = Path("../embedding/freq snap"
)

output_dir = Path("igraph_graphs")
output_dir.mkdir(exist_ok=True)

percentile = 99

print("[START] Costruzione grafi igraph con freq nodo")

for snap in range(1, 11):
    t0 = time.time()

    edges_tsv = edges_dir / f"edges_snap{snap}_p{percentile}.tsv"
    vec_file = vec_dir / f"fasttext_snap{snap}_filt.vec"
    freq_file = freq_dir / f"vocab_freq_snap{snap}.txt"
    out_graph = output_dir / f"graph_snap{snap}_p{percentile}.igp"

    print(f"\n[SNAPSHOT {snap}]")

    # archi
    df_edges = pd.read_csv(edges_tsv, sep="\t")

    src = df_edges.iloc[:, 0].astype(int).to_numpy()
    dst = df_edges.iloc[:, 1].astype(int).to_numpy()

    has_weight = df_edges.shape[1] > 2
    if has_weight:
        weights = df_edges.iloc[:, 2].astype(float).to_numpy()

    #token
    tokens = []
    with open(vec_file, encoding="utf-8") as f:
        next(f)
        for line in f:
            tokens.append(line.split()[0])

    n_nodes = len(tokens)

    # freq
    freq_map = {}
    with open(freq_file, encoding="utf-8") as f:
        for line in f:
            token, freq = line.split()
            freq_map[token] = int(freq)

    # allineamento token → freq (0 se assente)
    freqs = [freq_map.get(tok, 0) for tok in tokens]

    # grafo
    g = ig.Graph(n=n_nodes, directed=False)

    g.vs["token"] = tokens
    g.vs["freq"] = freqs

    g.add_edges(zip(src, dst))

    if has_weight:
        g.es["weight"] = weights

    # salvo
    g.write_pickle(out_graph)

    print(
        f"[DONE] nodi={n_nodes}, archi={g.ecount()}, "
        f"freq=yes, peso={'sì' if has_weight else 'no'} "
        f"in {time.time() - t0:.2f}s"
    )

print("\n[TOTAL] Tutti i grafi salvati")

In [ ]:
# per importare:
import igraph as ig

g = ig.Graph.Read_Pickle("igraph_graphs/graph_snap3_p99.igp")

print(g.vs[0]["token"], g.vs[0]["freq"])
print(g.vs[100]["token"], g.vs[100]["freq"])

print(g.es[0]["weight"])
